In [1]:
%cd /home/brimmann/works/xRAG

/home/brimmann/works/xRAG


/home/brimmann/works/xRAG/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
import torch
import torch.nn.functional as F
from torch.optim import Adam
from transformers import AutoTokenizer

# Import your custom model and config classes
from src.model.xMistral.modeling_xmistral import XMistralForCausalLM, XMistralConfig
from src.distill.modeling_xgemma import XGemmaForCausalLM, XGemmaConfig

# --- Configuration for the Validation Run ---
teacher_base_model_name = "mistralai/Mistral-7B-v0.1"
student_base_model_name = "google/gemma-2-2b"
retriever_hidden_size = 128  # An arbitrary dimension for this test
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"--- Validation Setup ---")
print(f"Device: {device}")
print(f"Mock Teacher Base: {teacher_base_model_name}")
print(f"Student Base: {student_base_model_name}")

--- Validation Setup ---
Device: cuda
Mock Teacher Base: mistralai/Mistral-7B-v0.1
Student Base: google/gemma-2-2b


In [ ]:
# We will use the student's tokenizer for both models in this test
from accelerate.utils import offload


tokenizer = AutoTokenizer.from_pretrained(student_base_model_name)
xrag_token = "<xRAG>"
tokenizer.add_special_tokens({"additional_special_tokens": [xrag_token]})
xrag_token_id = tokenizer.convert_tokens_to_ids(xrag_token)

# --- Configure Teacher ---
print("\nConfiguring mock teacher...")
teacher_config = XMistralConfig.from_pretrained(
    teacher_base_model_name,
    retriever_hidden_size=retriever_hidden_size,
    projector_type='mlp2x_gelu',
)
teacher_model = XMistralForCausalLM.from_pretrained(teacher_base_model_name, config=teacher_config, device_map="auto", offload_folder="offload_teacher")
teacher_model.resize_token_embeddings(len(tokenizer)) # Adjust for Gemma's tokenizer
teacher_model.set_xrag_token_id(xrag_token_id)
# teacher_model.to(device)
teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False
print("Mock teacher configured and frozen.")

# --- Configure Student ---
print("\nConfiguring student...")
student_config = XGemmaConfig.from_pretrained(
    student_base_model_name,
    retriever_hidden_size=retriever_hidden_size,
    projector_type='mlp2x_gelu'
)
student_model = XGemmaForCausalLM.from_pretrained(student_base_model_name, config=student_config, device_map="auto", offload_folder="offload_student")
student_model.resize_token_embeddings(len(tokenizer))
student_model.set_xrag_token_id(xrag_token_id)
# student_model.to(device)
student_model.train()
for name, param in student_model.named_parameters():
    if "projector" not in name:
        param.requires_grad = False
print("Student configured. Base model frozen.")

# --- Verify Trainable Parameters ---
print("\nTrainable parameters in student model:")
trainable_params = [name for name, param in student_model.named_parameters() if param.requires_grad]
if trainable_params:
    for name in trainable_params:
        print(name)
else:
    print("Warning: No trainable parameters found in student model!")


Configuring mock teacher...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of XMistralForCausalLM were not initialized from the model checkpoint at mistralai/Mistral-7B-v0.1 and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


: 